In [ ]:
#Paquetes a instalar
!pip install lime
!pip install -q -U datasets pandas pyarrow
!pip install -q transformers accelerate evaluate lime

# Carga de datos
from google.colab import drive
from datasets import load_dataset
from datasets import load_from_disk

# Manipulación y visualización
import pandas as pd
pd.set_option('display.max_colwidth', None)

import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

# Análisis de distribución de clases
from collections import Counter

# PyTorch
import torch
import torch.nn as nn

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Modelo preentrenado
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# Explicabilidad
from lime.lime_text import LimeTextExplainer

import lime.lime_tabular

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 25.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=a082b06ca7b82dc750290eed7c84e00a86db14a1c20447c7c235cc072d10c8c1
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26

In [ ]:
dataset = None

# Attempt 1: load from the unmerged "Convert dataset to Parquet" PR branch
# on PolyAI/banking77 (refs/pr/6), which already replaced the legacy
# loading script with train/test parquet files.
try:
    dataset = load_dataset("PolyAI/banking77", revision="refs/pr/6")
    print("Loaded from PolyAI/banking77 (refs/pr/6)")
except Exception as e:
    print(f"Attempt 1 failed: {e}")

# Attempt 2: fall back to a clean, already-standard-format mirror
if dataset is None:
    try:
        dataset = load_dataset("DeepPavlov/banking77")
        print("Loaded from DeepPavlov/banking77 mirror")
    except Exception as e:
        print(f"Attempt 2 failed: {e}")

# Attempt 3: last resort — pull the parquet files directly with pandas,
# bypassing `datasets` loading logic entirely.
if dataset is None:
    base = "https://huggingface.co/datasets/PolyAI/banking77/resolve/refs%2Fpr%2F6/data"
    train_df = pd.read_parquet(f"{base}/train-00000-of-00001.parquet")
    test_df = pd.read_parquet(f"{base}/test-00000-of-00001.parquet")
    print("Loaded via direct parquet download with pandas")
    print(train_df.head())

README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  295kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 93.0kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

Loaded from PolyAI/banking77 (refs/pr/6)


In [ ]:
# Previsualizamos con pandas
df = dataset["train"].to_pandas()
df.sample(5)

,text,label
287,What do I do if I already had a card with you guys?,13
9326,I just got my new card. How can I activate it?,0
3376,My cash deposit from a week ago is still not in my account. Can you help me please?,6
1651,What are the age limits for your service?,1
4138,My preference is Mastercard.,73


In [ ]:
# Obtenemos el tamaño del dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [ ]:
# Obtenemos los nombres de cada label
label_feature = dataset["train"].features["label"]
label_names = label_feature.names

for i, name in enumerate(label_names):
    print(i, name)

0 activate_my_card
1 age_limit
2 apple_pay_or_google_pay
3 atm_support
4 automatic_top_up
5 balance_not_updated_after_bank_transfer
6 balance_not_updated_after_cheque_or_cash_deposit
7 beneficiary_not_allowed
8 cancel_transfer
9 card_about_to_expire
10 card_acceptance
11 card_arrival
12 card_delivery_estimate
13 card_linking
14 card_not_working
15 card_payment_fee_charged
16 card_payment_not_recognised
17 card_payment_wrong_exchange_rate
18 card_swallowed
19 cash_withdrawal_charge
20 cash_withdrawal_not_recognised
21 change_pin
22 compromised_card
23 contactless_not_working
24 country_support
25 declined_card_payment
26 declined_cash_withdrawal
27 declined_transfer
28 direct_debit_payment_not_recognised
29 disposable_card_limits
30 edit_personal_details
31 exchange_charge
32 exchange_rate
33 exchange_via_app
34 extra_charge_on_statement
35 failed_transfer
36 fiat_currency_support
37 get_disposable_virtual_card
38 get_physical_card
39 getting_spare_card
40 getting_virtual_card
41 lost_o

# **1. TOKENIZACIÓN CON DISTILBERT**

Se seleccionó DistilBERT por ser un modelo Transformer preentrenado específicamente diseñado para tareas de procesamiento de lenguaje natural. En comparación con BERT, ofrece un menor costo computacional y tiempos de entrenamiento reducidos, manteniendo un rendimiento muy similar. Estas características lo convierten en una alternativa adecuada para adaptar mediante Transfer Learning y Fine-Tuning al problema de clasificación de consultas bancarias del dataset BANKING77.

In [ ]:
#Definimos modelo preentrenado
MODEL_NAME = "distilbert/distilbert-base-uncased"

MAX_LENGTH = 80

In [ ]:
#Cargamos tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
) #A diferencia de la tarea 2, ya no construimos nuestro propio vocabulario.

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
#Vemos cómo tokeniza DistilBERT
texto_ejemplo = dataset["train"][0]["text"]

print("Texto original:")
print(texto_ejemplo)

print("\nTokens:")
print(tokenizer.tokenize(texto_ejemplo))

Texto original:
I am still waiting on my card?

Tokens:
['i', 'am', 'still', 'waiting', 'on', 'my', 'card', '?']


In [ ]:
#Función de tokenización
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    ) #No ponemos padding acá porque lo haremos dinámicamente después.

# **2. SPLIT TRAIN / VALIDATION / TEST**

In [ ]:
#Split estratificado
from sklearn.model_selection import train_test_split

train_indices, val_indices = train_test_split(
    range(len(dataset["train"])),
    test_size=0.20,
    stratify=dataset["train"]["label"],
    random_state=42
)

train_dataset = dataset["train"].select(train_indices)

val_dataset = dataset["train"].select(val_indices)

test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 8002
Validation: 2001
Test: 3080


# **3. NOMBRES DE LAS CLASES**

In [ ]:
label_names = dataset["train"].features["label"].names

NUM_LABELS = len(label_names)

print("Cantidad de clases:", NUM_LABELS)

for i, label in enumerate(label_names):
    print(i, "-", label)

Cantidad de clases: 77
0 - activate_my_card
1 - age_limit
2 - apple_pay_or_google_pay
3 - atm_support
4 - automatic_top_up
5 - balance_not_updated_after_bank_transfer
6 - balance_not_updated_after_cheque_or_cash_deposit
7 - beneficiary_not_allowed
8 - cancel_transfer
9 - card_about_to_expire
10 - card_acceptance
11 - card_arrival
12 - card_delivery_estimate
13 - card_linking
14 - card_not_working
15 - card_payment_fee_charged
16 - card_payment_not_recognised
17 - card_payment_wrong_exchange_rate
18 - card_swallowed
19 - cash_withdrawal_charge
20 - cash_withdrawal_not_recognised
21 - change_pin
22 - compromised_card
23 - contactless_not_working
24 - country_support
25 - declined_card_payment
26 - declined_cash_withdrawal
27 - declined_transfer
28 - direct_debit_payment_not_recognised
29 - disposable_card_limits
30 - edit_personal_details
31 - exchange_charge
32 - exchange_rate
33 - exchange_via_app
34 - extra_charge_on_statement
35 - failed_transfer
36 - fiat_currency_support
37 - get_d

In [ ]:
id2label = {
    i: label
    for i, label in enumerate(label_names)
}

label2id = {
    label: i
    for i, label in enumerate(label_names)
}

In [ ]:
print(id2label[11])
print(label2id["card_arrival"])

card_arrival
11


# **4. TOKENIZAR TRAIN / VALIDATION / TEST**

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/8002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
) #Esto hace que dentro de cada batch las consultas se rellenen solo hasta la longitud necesaria.

# **5. CARGAR DISTILBERT**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ver si podemos con algo del código sacarlo pero no es importante

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# **7. TRANSFER LEARNING**

Primero vamos a hacer una primera etapa congelando DistilBERT.

In [ ]:
for param in model.distilbert.parameters():
    param.requires_grad = False

In [ ]:
#Analizamos cuántos parámetros entrenamos
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Parámetros totales:", total_params)
print("Parámetros entrenables:", trainable_params)

print(
    "Porcentaje entrenable:",
    round(
        100 * trainable_params / total_params,
        2
    ),
    "%"
)

Parámetros totales: 67012685
Parámetros entrenables: 649805
Porcentaje entrenable: 0.97 %


# **8. CONFIGURAR TRANSFER LEARNING**

In [ ]:
training_args_transfer = TrainingArguments(

    output_dir="./distilbert_transfer",

    num_train_epochs=3,

    learning_rate=5e-4,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    weight_decay=0.01,

    logging_strategy="epoch",

    save_total_limit=1,

    report_to="none",

    seed=42
)

Usamos eval_strategy="epoch" para evaluar al final de cada época y load_best_model_at_end=True para recuperar automáticamente el mejor checkpoint. Ambas opciones forman parte de la API actual de TrainingArguments.

In [ ]:
#Definimos el Trainer
trainer_transfer = Trainer(

    model=model,

    args=training_args_transfer,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
#Entrenamiento de Transfer Learning
trainer_transfer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,3.932009,3.365355,0.276362,0.269242,0.254825,0.210142
2,3.121971,2.788215,0.385307,0.463863,0.362509,0.329304
3,2.769218,2.614291,0.449775,0.457640,0.429182,0.401067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=753, training_loss=3.2743993913668246, metrics={'train_runtime': 49.2661, 'train_samples_per_second': 487.272, 'train_steps_per_second': 15.284, 'total_flos': 290977640018328.0, 'train_loss': 3.2743993913668246, 'epoch': 3.0})

In [ ]:
#Resultados sobre validación
transfer_val_results = trainer_transfer.evaluate()

transfer_val_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
2.769218,2.614291,3,0.449775,0.457640,0.429182,0.401067


{'eval_loss': 2.614290714263916,
 'eval_accuracy': 0.4497751124437781,
 'eval_precision_macro': 0.4576398459830094,
 'eval_recall_macro': 0.42918158802837114,
 'eval_f1_macro': 0.4010674438311579}

In [ ]:
#Guardamos métricas
transfer_results = {
    "Accuracy": transfer_val_results["eval_accuracy"],
    "Precision": transfer_val_results["eval_precision_macro"],
    "Recall": transfer_val_results["eval_recall_macro"],
    "F1": transfer_val_results["eval_f1_macro"]
}

transfer_results

{'Accuracy': 0.4497751124437781,
 'Precision': 0.4576398459830094,
 'Recall': 0.42918158802837114,
 'F1': 0.4010674438311579}

# **9. FINE-TUNING**

In [ ]:
for param in model.distilbert.parameters():
    param.requires_grad = True

Descongelamos DistilBERT.

In [ ]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Parámetros entrenables:", trainable_params)
print("Parámetros totales:", total_params)

print(
    "Porcentaje entrenable:",
    round(
        100 * trainable_params / total_params,
        2
    ),
    "%"
)

Parámetros entrenables: 67012685
Parámetros totales: 67012685
Porcentaje entrenable: 100.0 %


# **10. CONFIGURACIÓN FINE-TUNING**

In [ ]:
training_args_finetune = TrainingArguments(

    output_dir="./distilbert_finetuned",

    num_train_epochs=4,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    weight_decay=0.01,

    logging_strategy="epoch",

    save_total_limit=1,

    report_to="none",

    seed=42
)

In [ ]:
#Definimos el Trainer Fine-Tuning
trainer_finetune = Trainer(

    model=model,

    args=training_args_finetune,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
#Entrenamiento del Fine-Tune
trainer_finetune.train()

# **11. EVALUACIÓN FINAL SOBRE TEST**

In [ ]:
test_results = trainer_finetune.evaluate(
    test_tokenized
)

test_results

In [ ]:
print(
    "Accuracy:",
    round(test_results["eval_accuracy"], 4)
)

print(
    "Precision macro:",
    round(test_results["eval_precision_macro"], 4)
)

print(
    "Recall macro:",
    round(test_results["eval_recall_macro"], 4)
)

print(
    "F1 macro:",
    round(test_results["eval_f1_macro"], 4)
)

El entrenamiento exclusivo de la capa clasificadora produjo un rendimiento limitado, con un accuracy de aproximadamente 46,0 % y un F1-score macro de 0,416. Al permitir el ajuste de todos los parámetros mediante Fine-Tuning, el modelo logró adaptarse con mayor eficacia al dominio específico de BANKING77, alcanzando un accuracy de 90,9 % y un F1-score macro de 0,909.

# **12. PREDICCIONES**

In [ ]:
predictions = trainer_finetune.predict(
    test_tokenized
)

y_test = predictions.label_ids

y_pred = np.argmax(
    predictions.predictions,
    axis=-1
)

y_prob = torch.softmax(
    torch.tensor(predictions.predictions),
    dim=1
).numpy()

print(
    "Cantidad de observaciones:",
    len(y_test)
)

print(
    "Forma de probabilidades:",
    y_prob.shape
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    )
)

In [ ]:
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report_dict
).T

report_df

In [ ]:
class_results = report_df.iloc[:NUM_LABELS].copy()

class_results = class_results.sort_values(
    "f1-score"
)

class_results.head(10)

In [ ]:
history_df = pd.DataFrame(
    trainer_finetune.state.log_history
)

history_df

In [ ]:
train_history = history_df[
    history_df["loss"].notna()
].copy()

val_history = (
    history_df[
        history_df["eval_loss"].notna()
    ]
    .drop_duplicates(
        subset="epoch",
        keep="first"
    )
    .copy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    train_history["epoch"],
    train_history["loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    val_history["epoch"],
    val_history["eval_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Época")
plt.ylabel("Loss")

plt.title(
    "Evolución de la pérdida - Fine-Tuning"
)

plt.legend()

plt.show()

In [ ]:
print( val_history["eval_f1_macro"])

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    val_history["epoch"],
    val_history["eval_f1_macro"],
    marker="o"
)

plt.xlabel("Época")
plt.ylabel("F1 Macro")

plt.title(
    "F1 Macro de validación"
)

plt.show()

# **16. COMPARACIÓN CONTRA TAREA 2**

In [ ]:
transformer_scratch = {
    "Modelo": "Transformer desde cero",
    "Accuracy": 0.790,
    "Precision": 0.803,
    "Recall": 0.790,
    "F1": 0.790
}

distilbert_transfer = {
    "Modelo": "DistilBERT Transfer Learning",
    "Accuracy": transfer_results["Accuracy"],
    "Precision": transfer_results["Precision"],
    "Recall": transfer_results["Recall"],
    "F1": transfer_results["F1"]
}

distilbert_finetune = {
    "Modelo": "DistilBERT Fine-Tuning",
    "Accuracy": test_results["eval_accuracy"],
    "Precision": test_results["eval_precision_macro"],
    "Recall": test_results["eval_recall_macro"],
    "F1": test_results["eval_f1_macro"]
}

comparison_df = pd.DataFrame([
    transformer_scratch,
    distilbert_transfer,
    distilbert_finetune
])

comparison_df

In [ ]:
comparison_plot = comparison_df.set_index(
    "Modelo"
)

comparison_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)

plt.title(
    "Comparación de rendimiento entre modelos"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.show()

# **18. EXPLICABILIDAD CON LIME**

In [ ]:
def predict_proba(texts):

    # Define device: Use CUDA (GPU) if available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    model.to(device)
    model.eval()

    with torch.no_grad():

        outputs = model(
            **encoded
        )

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    return probabilities.cpu().numpy()

In [ ]:
explainer = LimeTextExplainer(
    class_names=label_names
)

# **19. EXPLICAR UNA PREDICCIÓN CORRECTA**

In [ ]:
correct_indices = np.where(
    y_test == y_pred
)[0]

indice_correcto = int(
    correct_indices[0]
)

texto_correcto = test_dataset[
    indice_correcto
]["text"]

real_correcto = y_test[
    indice_correcto
]

pred_correcto = y_pred[
    indice_correcto
]

print("Texto:")
print(texto_correcto)

print("\nClase real:")
print(label_names[real_correcto])

print("\nPredicción:")
print(label_names[pred_correcto])

In [ ]:
exp_correcto = explainer.explain_instance(
    texto_correcto,
    predict_proba,
    num_features=10,
    labels=[pred_correcto],
    num_samples=1000
)

In [ ]:
fig = exp_correcto.as_pyplot_figure(
    label=pred_correcto
)

plt.title(
    f"LIME - {label_names[pred_correcto]}"
)

plt.tight_layout()

plt.show()

In [ ]:
wrong_indices = np.where(
    y_test != y_pred
)[0]

print(
    "Cantidad de errores:",
    len(wrong_indices)
)

indice_error = int(
    wrong_indices[0]
)

texto_error = test_dataset[
    indice_error
]["text"]

real_error = y_test[
    indice_error
]

pred_error = y_pred[
    indice_error
]

print("Texto:")
print(texto_error)

print("\nClase real:")
print(label_names[real_error])

print("\nClase predicha:")
print(label_names[pred_error])

In [ ]:
exp_error = explainer.explain_instance(
    texto_error,
    predict_proba,
    num_features=10,
    labels=[pred_error],
    num_samples=1000
)

In [ ]:
fig = exp_error.as_pyplot_figure(
    label=pred_error
)

plt.title(
    f"LIME - Error: {label_names[pred_error]}"
)

plt.tight_layout()

plt.show()

In [ ]:
exp_error.as_list(
    label=pred_error
)

# PROYECTO FINAL — INTEGRACIÓN DE LA PRÁCTICA 3

> **Importante:** desde este punto se conserva todo el código desarrollado en la Práctica 3 y se agregan únicamente los componentes necesarios para convertirlo en el proyecto final solicitado.
>
> La solución final mantiene como núcleo experimental **BANKING77 + DistilBERT + Transfer Learning + Fine-Tuning + LIME**, y agrega la definición formal del proyecto, análisis del ciclo de vida y un prototipo funcional.


## 20. Planteamiento del proyecto

### Problema

Clasificar automáticamente consultas bancarias escritas en lenguaje natural dentro de una de las 77 intenciones de BANKING77.

### Propuesta de solución

Desarrollar un clasificador basado en **DistilBERT Fine-Tuned**, aprovechando las representaciones lingüísticas aprendidas durante el preentrenamiento y adaptándolas al dominio bancario.

### Usuario / caso de uso

Un operador o sistema de atención al cliente introduce una consulta y recibe:

- intención predicha;
- probabilidad/confianza;
- principales alternativas;
- explicación de la predicción mediante LIME.

### ¿Por qué Deep Learning?

Las consultas pueden expresar una misma intención con vocabulario y estructuras diferentes. Un modelo basado en Transformer permite capturar relaciones contextuales que serían difíciles de representar mediante reglas o enfoques puramente basados en palabras clave.

### Arquetipo de producto

Depende del contexto, en nuestro caso podrían ser cualquiera de los tres.

### Impacto vs. factibilidad

- **Impacto:** alto, porque permite automatizar una tarea repetitiva y acelerar el direccionamiento de consultas.
- **Factibilidad:** alta para un prototipo, porque BANKING77 proporciona datos etiquetados y existe un modelo Transformer preentrenado.


## 21. Ciclo de vida aplicado

| Fase | Implementación en el proyecto |
|---|---|
| Planificación | Definición del problema, usuario, objetivo y métricas |
| Recolección / datos | Dataset BANKING77 |
| Preparación | Exploración, split, tokenización, padding y embedding |
| Exploración | Distribución de clases y análisis de ejemplos |
| Modelo inicial | DistilBERT preentrenado |
| Transfer Learning | Congelación del backbone y entrenamiento de la cabeza |
| Refinamiento | Fine-Tuning de DistilBERT |
| Evaluación | Accuracy, Precision, Recall, F1 y errores |
| Explicabilidad | LIME |
| Prototipo | Interfaz Gradio |
| Producción | Monitoreo, revisión humana, privacidad y reentrenamiento |


## 22. Consolidación de resultados de la Práctica 3

Esta sección utiliza **las variables y resultados calculados por el código original**, en lugar de volver a entrenar un modelo diferente.

Los objetos principales heredados de la práctica son:

- `trainer_transfer`
- `trainer_finetune`
- `transfer_results`
- `test_results`
- `y_test`
- `y_pred`
- `report_df`
- `comparison_df`
- `predict_proba`
- `explainer`
- `exp_correcto`
- `exp_error`

De esta manera, el proyecto final queda directamente conectado con el trabajo realizado en la Práctica 3.


In [ ]:
# Resumen de resultados obtenidos en la Práctica 3

final_results = {
    "Accuracy": test_results.get("eval_accuracy", np.nan),
    "Precision Macro": test_results.get("eval_precision_macro", np.nan),
    "Recall Macro": test_results.get("eval_recall_macro", np.nan),
    "F1 Macro": test_results.get("eval_f1_macro", np.nan),
}

results_summary = pd.DataFrame(
    [final_results],
    index=["DistilBERT Fine-Tuning"]
)

display(results_summary.round(4))


## 23. Comparación de enfoques

La comparación es parte importante de la justificación técnica.

Se mantienen los modelos comparados en la Práctica 3 y se presenta el resultado final de forma consolidada.

El criterio principal para este problema multiclase es **F1 Macro**, acompañado por Accuracy.


In [ ]:
# Mostrar la comparación que ya fue construida en la Práctica 3

try:
    display(comparison_df)
except NameError:
    print("comparison_df no está disponible. Ejecutá las celdas anteriores de la Práctica 3.")


## 24. Análisis de errores para el proyecto final

Además de la métrica global, analizamos las predicciones incorrectas para identificar dónde el modelo presenta dificultades.

Esto permite discutir el problema desde una perspectiva de ciclo de vida: los errores pueden indicar la necesidad de más datos, clases mejor diferenciadas, ejemplos adicionales o un mecanismo de revisión humana.


In [ ]:
# Tabla de errores utilizando las predicciones ya calculadas en la Práctica 3

test_texts = test_dataset["text"]

error_analysis = pd.DataFrame({
    "text": test_texts,
    "real": [id2label[int(i)] for i in y_test],
    "prediccion": [id2label[int(i)] for i in y_pred]
})

error_analysis["correcto"] = error_analysis["real"] == error_analysis["prediccion"]

errors_only = error_analysis[~error_analysis["correcto"]].copy()

print(f"Cantidad total de ejemplos: {len(error_analysis)}")
print(f"Cantidad de errores: {len(errors_only)}")
print(f"Accuracy: {error_analysis['correcto'].mean():.4f}")

display(errors_only.head(20))


## 25. Prototipo funcional

El siguiente bloque utiliza **el mismo `model`, `tokenizer`, `id2label` y `predict_proba` de la Práctica 3**.

No se crea un segundo modelo ni se vuelve a entrenar.

El usuario podrá introducir una consulta y obtener la intención predicha junto con las probabilidades de las cinco clases principales.


In [ ]:
# Función de predicción para el prototipo

def classify_query(text, top_k=5):
    probabilities = predict_proba([text])[0]

    top_indices = np.argsort(probabilities)[::-1][:top_k]

    result = pd.DataFrame({
        "Intención": [id2label[int(i)] for i in top_indices],
        "Probabilidad": [float(probabilities[i]) for i in top_indices]
    })

    return result


# Prueba rápida
consulta_demo = "My card hasn't arrived yet"

resultado_demo = classify_query(consulta_demo)

print("Consulta:", consulta_demo)
display(resultado_demo)


In [ ]:
# Explicabilidad del ejemplo del prototipo

predicted_class = int(np.argmax(predict_proba([consulta_demo])[0]))

explanation_demo = explainer.explain_instance(
    consulta_demo,
    predict_proba,
    num_features=10,
    top_labels=1
)

print("Consulta:", consulta_demo)
print("Predicción:", id2label[predicted_class])
print()
print("Palabras con mayor contribución:")
display(
    pd.DataFrame(
        explanation_demo.as_list(label=predicted_class),
        columns=["Palabra", "Peso"]
    )
)


## 26. Interfaz Gradio

Esta interfaz constituye la demostración funcional del proyecto.

Flujo de uso:

**Consulta del usuario → DistilBERT → probabilidades → intención → explicación**

Para abrir la interfaz, ejecutar la última línea de la celda.


In [ ]:
# Instalación de Gradio (si todavía no está instalado)
!pip -q install gradio


In [ ]:
import gradio as gr

def app_predict(text):
    if not text or not text.strip():
        return pd.DataFrame(columns=["Intención", "Probabilidad"]), "Ingrese una consulta."

    probabilities = predict_proba([text])[0]
    top_indices = np.argsort(probabilities)[::-1][:5]

    table = pd.DataFrame({
        "Intención": [id2label[int(i)] for i in top_indices],
        "Probabilidad": [float(probabilities[i]) for i in top_indices]
    })

    predicted_class = int(top_indices[0])

    explanation = explainer.explain_instance(
        text,
        predict_proba,
        num_features=8,
        top_labels=1
    )

    explanation_text = "\n".join(
        [f"{word}: {weight:+.3f}" for word, weight in explanation.as_list(label=predicted_class)]
    )

    output_text = (
        f"Intención predicha: {id2label[predicted_class]}\n"
        f"Confianza: {probabilities[predicted_class]:.2%}\n\n"
        f"Contribución de palabras (LIME):\n{explanation_text}"
    )

    return table, output_text


demo = gr.Interface(
    fn=app_predict,
    inputs=gr.Textbox(
        label="Consulta bancaria",
        placeholder="Ejemplo: My card hasn't arrived yet",
        lines=3
    ),
    outputs=[
        gr.Dataframe(
            headers=["Intención", "Probabilidad"],
            label="Top 5 intenciones"
        ),
        gr.Textbox(label="Explicación")
    ],
    title="Banking77 — Clasificador de intenciones",
    description="DistilBERT Fine-Tuned + LIME",
    examples=[
        ["My card hasn't arrived yet"],
        ["I forgot my PIN"],
        ["I need help with a transfer"],
        ["Why was I charged a fee?"]
    ]
)

# Ejecutar para abrir la aplicación:
demo.launch(share=True)


## 27. Consideraciones para producción

El prototipo demuestra la funcionalidad principal, pero un despliegue real requeriría:

### Monitoreo
- Accuracy/F1 cuando exista feedback etiquetado.
- porcentaje de predicciones con baja confianza;
- distribución de intenciones;
- cambios en la distribución de consultas.

### Human in the Loop
Las predicciones con baja confianza podrían derivarse a un operador. Las correcciones realizadas por operadores podrían incorporarse posteriormente como nuevos datos etiquetados.

### Privacidad
En un entorno bancario real, las consultas pueden contener información sensible. Serían necesarias políticas de anonimización, almacenamiento seguro y control de acceso.

### Reentrenamiento
El lenguaje y las necesidades de los clientes pueden cambiar. El modelo debería reevaluarse periódicamente y eventualmente reentrenarse con nuevos ejemplos.

### Escalabilidad
El prototipo podría exponerse como API y optimizarse mediante batching, infraestructura adecuada y monitoreo de latencia.


## 28. Conclusiones

La solución final integra el trabajo realizado en la Práctica 3 dentro de un proyecto completo de Deep Learning.

El modelo utiliza:

- BANKING77;
- DistilBERT;
- Transfer Learning;
- Fine-Tuning;
- evaluación cuantitativa;
- análisis de errores;
- LIME;
- un prototipo interactivo.

La evolución principal respecto de la práctica consiste en pasar de un experimento de modelado a una solución orientada a un caso de uso, incorporando problema, usuario, ciclo de vida, explicabilidad y demostración funcional.
